# Mutual Fund Performance Analytics
This notebook computes key performance metrics for 40 mutual funds, including:
- Daily Returns, CAGR (1yr, 3yr, 5yr)
- Risk Metrics (Sharpe Ratio, Sortino Ratio, Maximum Drawdown)
- Alpha & Beta (OLS regression)
- Fund Scorecard (0-100 scale)
- Benchmark comparison

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

conn = sqlite3.connect('bluestock_mf.db')

# 1. Load Data
nav_df = pd.read_sql("SELECT * FROM fact_nav", conn)
nav_df['date'] = pd.to_datetime(nav_df['date'])
nav_df = nav_df.sort_values(['amfi_code', 'date'])

fund_df = pd.read_sql("SELECT * FROM dim_fund", conn)

bench_df = pd.read_sql("SELECT * FROM benchmark_indices", conn)
bench_df['date'] = pd.to_datetime(bench_df['date'])

## 1. Compute Daily Returns & Benchmarks

In [ ]:
nav_df['daily_return'] = nav_df.groupby('amfi_code')['nav'].pct_change()

nifty100_df = bench_df[bench_df['index_name'].str.contains('100')].sort_values('date')
nifty100_df = nifty100_df.drop_duplicates(subset=['date'], keep='last')
nifty100_df['daily_return'] = nifty100_df['close_value'].pct_change()
nifty100_returns = nifty100_df.set_index('date')['daily_return']

nifty50_df = bench_df[bench_df['index_name'] == 'NIFTY50'].sort_values('date')
nifty50_df = nifty50_df.drop_duplicates(subset=['date'], keep='last')
nifty50_returns = nifty50_df.set_index('date')['close_value']

## 2. Compute Metrics (CAGR, Sharpe, Sortino, Alpha, Beta, Max Drawdown)

In [ ]:
def get_cagr(start_val, end_val, years):
    if years <= 0 or pd.isna(start_val) or pd.isna(end_val) or start_val == 0:
        return np.nan
    return (end_val / start_val) ** (1 / years) - 1

rf = 0.065
max_date = nav_df['date'].max()
funds = nav_df['amfi_code'].unique()
results = []

for amfi in funds:
    fund_data = nav_df[nav_df['amfi_code'] == amfi].copy()
    fund_data = fund_data.drop_duplicates(subset=['date'], keep='last')
    if len(fund_data) < 2: continue
    
    end_val = fund_data.iloc[-1]['nav']
    end_date = fund_data.iloc[-1]['date']
    
    # Dates for CAGR
    date_1y = end_date - pd.DateOffset(years=1)
    date_3y = end_date - pd.DateOffset(years=3)
    date_5y = end_date - pd.DateOffset(years=5)
    
    nav_1y = fund_data[fund_data['date'] <= date_1y]
    nav_3y = fund_data[fund_data['date'] <= date_3y]
    nav_5y = fund_data[fund_data['date'] <= date_5y]
    
    start_1y = nav_1y.iloc[-1]['nav'] if not nav_1y.empty else np.nan
    start_3y = nav_3y.iloc[-1]['nav'] if not nav_3y.empty else np.nan
    start_5y = nav_5y.iloc[-1]['nav'] if not nav_5y.empty else np.nan
    
    cagr_1y = get_cagr(start_1y, end_val, 1)
    cagr_3y = get_cagr(start_3y, end_val, 3)
    cagr_5y = get_cagr(start_5y, end_val, 5)
    
    daily_returns = fund_data.set_index('date')['daily_return'].dropna()
    mean_ret = daily_returns.mean() * 252
    std_ret = daily_returns.std() * np.sqrt(252)
    sharpe = (mean_ret - rf) / std_ret if std_ret != 0 else np.nan
    
    downside_returns = daily_returns[daily_returns < 0]
    downside_std = downside_returns.std() * np.sqrt(252)
    sortino = (mean_ret - rf) / downside_std if downside_std != 0 and not pd.isna(downside_std) else np.nan
    
    aligned = pd.concat([daily_returns, nifty100_returns], axis=1, join='inner').dropna()
    aligned.columns = ['fund', 'bench']
    if len(aligned) > 10:
        slope, intercept, r, p, se = stats.linregress(aligned['bench'], aligned['fund'])
        beta = slope
        alpha = intercept * 252
    else:
        beta = np.nan
        alpha = np.nan
        
    fund_data['running_max'] = fund_data['nav'].cummax()
    fund_data['drawdown'] = fund_data['nav'] / fund_data['running_max'] - 1
    max_dd = fund_data['drawdown'].min()
    
    exp_ratio = fund_df.loc[fund_df['amfi_code'] == amfi, 'expense_ratio_pct'].values
    expense = exp_ratio[0] if len(exp_ratio) > 0 else np.nan
    
    results.append({
        'amfi_code': amfi,
        'cagr_1y': cagr_1y,
        'cagr_3y': cagr_3y,
        'cagr_5y': cagr_5y,
        'sharpe': sharpe,
        'sortino': sortino,
        'alpha': alpha,
        'beta': beta,
        'max_dd': max_dd,
        'expense_ratio': expense
    })

res_df = pd.DataFrame(results)

## 3. Fund Scorecard (0-100) & Saving CSVs
Composite Score = 30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × expense ratio rank (inverse) + 10% × max DD rank (inverse)

In [ ]:
res_df['rank_3yr'] = res_df['cagr_3y'].rank(pct=True) * 100
res_df['rank_sharpe'] = res_df['sharpe'].rank(pct=True) * 100
res_df['rank_alpha'] = res_df['alpha'].rank(pct=True) * 100
res_df['rank_expense'] = res_df['expense_ratio'].rank(pct=True, ascending=False) * 100
res_df['rank_maxdd'] = res_df['max_dd'].rank(pct=True, ascending=True) * 100

res_df['Score'] = (0.30 * res_df['rank_3yr'].fillna(0) +
                   0.25 * res_df['rank_sharpe'].fillna(0) +
                   0.20 * res_df['rank_alpha'].fillna(0) +
                   0.15 * res_df['rank_expense'].fillna(0) +
                   0.10 * res_df['rank_maxdd'].fillna(0))

alpha_beta_df = res_df[['amfi_code', 'alpha', 'beta']].copy()
alpha_beta_df = alpha_beta_df.merge(fund_df[['amfi_code', 'scheme_name']], on='amfi_code', how='left')
alpha_beta_df.to_csv('alpha_beta.csv', index=False)

scorecard_df = res_df[['amfi_code', 'Score', 'cagr_1y', 'cagr_3y', 'sharpe', 'sortino', 'alpha', 'beta', 'max_dd', 'expense_ratio']].copy()
scorecard_df = scorecard_df.merge(fund_df[['amfi_code', 'scheme_name']], on='amfi_code', how='left')
scorecard_df = scorecard_df.sort_values('Score', ascending=False)
scorecard_df.to_csv('fund_scorecard.csv', index=False)

scorecard_df.head()

## 4. Benchmark Comparison Chart & Tracking Error

In [ ]:
top5 = scorecard_df.head(5)
plt.figure(figsize=(14, 8))
start_date_3y = max_date - pd.DateOffset(years=3)

for _, row in top5.iterrows():
    amfi = row['amfi_code']
    f_data = nav_df[(nav_df['amfi_code'] == amfi) & (nav_df['date'] >= start_date_3y)].sort_values('date')
    if len(f_data) > 0:
        norm_nav = f_data['nav'] / f_data.iloc[0]['nav'] * 100
        plt.plot(f_data['date'], norm_nav, label=row['scheme_name'])

n100_3y = bench_df[(bench_df['index_name'].str.contains('100')) & (bench_df['date'] >= start_date_3y)].sort_values('date')
if len(n100_3y) > 0:
    plt.plot(n100_3y['date'], n100_3y['close_value'] / n100_3y.iloc[0]['close_value'] * 100, label='Nifty 100', color='black', linewidth=2, linestyle='--')

n50_3y = bench_df[(bench_df['index_name'] == 'NIFTY50') & (bench_df['date'] >= start_date_3y)].sort_values('date')
if len(n50_3y) > 0:
    plt.plot(n50_3y['date'], n50_3y['close_value'] / n50_3y.iloc[0]['close_value'] * 100, label='Nifty 50', color='grey', linewidth=2, linestyle=':')

plt.title('Top 5 Funds by Score vs Benchmarks (3 Years Rebased to 100)')
plt.xlabel('Date')
plt.ylabel('Rebased Value')
plt.legend()
plt.tight_layout()
plt.savefig('benchmark_comparison_chart.png')
plt.show()

# Tracking Error
print("Tracking Error vs Nifty 50:")
for _, row in top5.iterrows():
    amfi = row['amfi_code']
    f_data = nav_df[(nav_df['amfi_code'] == amfi)].drop_duplicates(subset=['date']).set_index('date')['daily_return']
    aligned = pd.concat([f_data, nifty50_df.set_index('date')['close_value'].pct_change()], axis=1, join='inner').dropna()
    aligned.columns = ['fund', 'bench']
    tracking_err = (aligned['fund'] - aligned['bench']).std() * np.sqrt(252)
    print(f"{row['scheme_name']}: {tracking_err:.4f}")